In [3]:
import pandas as pd
import os

filepath = "../DATA-HTML-STOCK/NEPSECompany/NEPSECompanyExtractor.csv"
df = pd.read_csv(filepath)
symbols = df['Symbol'].tolist()

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]

def detailed_prediction(score):
    if score >= 0.60:
        return "Strongly Positive"
    elif score >= 0.20:
        return "Positive"
    elif score >= 0.05:
        return "Slightly Positive"
    elif score > -0.05:
        return "Neutral"
    elif score > -0.20:
        return "Slightly Negative"
    elif score > -0.60:
        return "Negative"
    else:
        return "Strongly Negative"

def check_existing_semifinal(savepath):
    saved_date = None
    old_df = None
    if os.path.exists(savepath):
        old_df = pd.read_csv(savepath)
        saved_date = pd.to_datetime(old_df["Date_Stock"].iloc[-1])
    return saved_date, old_df

for symbol in symbols:
    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    stockpath = f"../DATA-HTML-STOCK/NEPSEDATA/{symbol}.csv"
    if not os.path.exists(stockpath):
        print(f"no stock {symbol} found ignored it")
        continue

    stock_df = pd.read_csv(stockpath, low_memory=False)
    stock_df[["Open", "High", "Low", "Close"]] = stock_df[["Open", "High", "Low", "Close"]].apply(pd.to_numeric, errors='coerce').fillna(0)
    stock_close_plus_date = stock_df[["Date", "Close"]]

    savepath = f"../DATA-HTML-STOCK/SemiFinalDataset/{symbol}.csv"
    saved_date, old_df = check_existing_semifinal(savepath)

    if saved_date is not None:
        stock_close_plus_date = stock_close_plus_date[pd.to_datetime(stock_close_plus_date["Date"]) > saved_date]
        if stock_close_plus_date.empty:
            print(f"{symbol} no new stock data found")
            continue

    print(stock_close_plus_date.head())

    if symbol in stock_no_news:
        print(f"\n{symbol} is in stock no news list!\n")
        new_df = stock_close_plus_date.rename(columns={"Date": "Date_Stock"})
        new_df["Date_News"] = None
        new_df["Sentiment_Score"] = 0.0
        new_df["Prediction"] = "Neutral"
        new_df = new_df[["Date_Stock", "Close", "Date_News", "Sentiment_Score", "Prediction"]]
        if old_df is not None:
            new_df = pd.concat([old_df, new_df], ignore_index=True)
        new_df.to_csv(savepath, index=False)
        print(new_df)
        continue

    newspath = f"../DATA-HTML-STOCK/STOCKSENTIMENT/{symbol}news_sentiment.csv"
    if not os.path.exists(newspath):
        print(f"no news {symbol} file found")
        new_df = stock_close_plus_date.rename(columns={"Date": "Date_Stock"})
        new_df["Date_News"] = None
        new_df["Sentiment_Score"] = 0.0
        new_df["Prediction"] = "Neutral"
        new_df = new_df[["Date_Stock", "Close", "Date_News", "Sentiment_Score", "Prediction"]]
        if old_df is not None:
            new_df = pd.concat([old_df, new_df], ignore_index=True)
        new_df.to_csv(savepath, index=False)
        print(new_df)
        continue

    sentiment_df = pd.read_csv(newspath)
    sentiment_plus_datas = sentiment_df[['Date', 'Sentiment_Score']]

    if saved_date is not None:
        sentiment_plus_datas = sentiment_plus_datas[pd.to_datetime(sentiment_plus_datas["Date"]) > saved_date]

    print(sentiment_plus_datas.head())

    new_stock_df = stock_close_plus_date.rename(columns={"Date": "Date_Stock"})
    news_df = sentiment_plus_datas.rename(columns={"Date": "Date_News"})

    news_df = news_df.groupby("Date_News").agg({
        "Sentiment_Score": "mean"
    }).reset_index()

    news_df["Prediction"] = news_df["Sentiment_Score"].apply(detailed_prediction)

    new_df = pd.merge(new_stock_df, news_df, left_on="Date_Stock", right_on="Date_News", how="left")

    new_df["Sentiment_Score"] = new_df["Sentiment_Score"].fillna(0)
    new_df["Prediction"] = new_df["Prediction"].fillna("Neutral")
    new_df["Close"] = stock_close_plus_date['Close'].values

    new_df = new_df[["Date_Stock", "Close", "Date_News", "Sentiment_Score", "Prediction"]]

    if old_df is not None:
        new_df = pd.concat([old_df, new_df], ignore_index=True)

    print(new_df)
    new_df.to_csv(savepath, index=False)
print("All Done")

         Date  Close
0  2026-03-03  305.0
1  2026-03-01  302.6
2  2026-02-26  298.0
3  2026-02-25  294.6
4  2026-02-24  293.0
         Date  Sentiment_Score
0  2026-02-17         0.900208
1  2026-02-12         0.021785
2  2026-01-14         0.014030
3  2025-12-15         0.015679
4  2025-12-11         0.071837
      Date_Stock  Close Date_News  Sentiment_Score Prediction
0     2026-03-03  305.0       NaN              0.0    Neutral
1     2026-03-01  302.6       NaN              0.0    Neutral
2     2026-02-26  298.0       NaN              0.0    Neutral
3     2026-02-25  294.6       NaN              0.0    Neutral
4     2026-02-24  293.0       NaN              0.0    Neutral
...          ...    ...       ...              ...        ...
3523  2010-09-12  118.0       NaN              0.0    Neutral
3524  2010-09-09  122.0       NaN              0.0    Neutral
3525  2010-09-08  125.0       NaN              0.0    Neutral
3526  2010-09-07  138.0       NaN              0.0    Neutral
3527  